# 01 · Exploratory data analysis: MVTec AD (bottle, carpet, screw)

This notebook looks at the data **before any model is trained**. It answers four questions:

1. How many images are there, and how are they split?
2. What size and colour format are the images?
3. What do good parts and defects look like?
4. How large are the defects, and is 256 × 256 pixels enough to see them?

The findings are written up in [`docs/data_card.md`](../docs/data_card.md).

**How to run it**

1. Build the manifest first: `uv run dvc pull` then `uv run dvc repro` (this creates `data/processed/manifest.parquet`).
2. In VS Code, pick the project's `.venv` Python as the kernel (top right of the notebook).
3. Run all cells.

**Rules for notebooks in this project**

- Notebooks are for **exploring**. Anything reusable lives in `src/anomaly_inspector/` with tests.
- Outputs are removed automatically at commit time by the `nbstripout` pre-commit hook, so the git history stays small and readable.

## 1. Setup

The manifest stores image paths **relative to the repo root** (so they also work in Docker and CI). A notebook starts in the `notebooks/` folder, so the first cell moves to the repo root, found by looking for `dvc.yaml`.

In [ ]:
import hashlib
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from anomaly_inspector.preprocessing import MEAN, STD, preprocess

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dvc.yaml").exists())
os.chdir(ROOT)
CATEGORIES = ["bottle", "carpet", "screw"]
SPLITS = ["train", "val_normal", "calib", "holdout"]
print("Working directory:", Path.cwd())

## 2. The manifest: one row per image

The `prepare` stage (Phase 1, step 7) scanned the dataset and wrote this table. The rest of the project reads the manifest instead of scanning folders, so everyone works with exactly the same list of images and the same splits.

| Column | Meaning |
| --- | --- |
| `path` | image file, relative to the repo root |
| `category` | the product: bottle, carpet or screw |
| `source_split` | where MVTec put it: `train` (good only) or `test` |
| `defect_type` | `good`, or the name of the defect |
| `label` | 0 = good, 1 = defective |
| `mask_path` | the pixel mask of the defect (empty for good images) |
| `width`, `height`, `mode` | image size and colour mode (`RGB` or `L` = grayscale) |
| `sha1` | fingerprint of the file, used to detect duplicates |
| `split` | our split: `train`, `val_normal`, `calib` or `holdout` |

In [ ]:
manifest = pd.read_parquet("data/processed/manifest.parquet")
print(manifest.shape)
manifest.head()

## 3. How many images are there?

MVTec AD gives a `train` folder with **good images only** and a `test` folder with good and defective images. The model learns what "normal" looks like from good images alone (one-class anomaly detection), so defects never appear in training.

In [ ]:
pd.crosstab(manifest["category"], manifest["source_split"], margins=True)

Our `prepare` stage then cut these into four splits:

- **train** (90% of train/good): what PatchCore stores and STFPM learns from.
- **val_normal** (10% of train/good): good images held back for early stopping and for a threshold that needs no defects.
- **calib** (half of test): every decision, such as the threshold and the hyperparameters, is made here.
- **holdout** (the other half of test): used once, for the numbers we report.

The table counts good and defective images in each split.

In [ ]:
kind = manifest["label"].map({0: "good", 1: "defective"})
pd.crosstab([manifest["category"], kind], manifest["split"])[SPLITS]

Each defect type was split in half between `calib` and `holdout` (stratified), so both halves contain every kind of defect:

In [ ]:
test = manifest[manifest["source_split"] == "test"]
pd.crosstab([test["category"], test["defect_type"]], test["split"])

### The simulation pool

Before DVC recorded the data, 20% of each category's train/good images were moved to `data/simulation/`. Phase 9 uses them as "new parts from the factory line", so they must **never** be in the manifest. The check below compares file fingerprints (SHA-1): an overlap would be data leakage.

In [ ]:
pool = sorted(Path("data/simulation").glob("*/good/*.png"))
print(pd.Series([p.parts[2] for p in pool]).value_counts().sort_index().to_dict())

pool_hashes = {hashlib.sha1(p.read_bytes()).hexdigest() for p in pool}
overlap = pool_hashes & set(manifest["sha1"])
print(f"{len(pool)} pool images, {len(overlap)} of them also in the manifest")

## 4. Image sizes and colour modes

`preprocess()` resizes every image to a square and converts it to 3-channel RGB, because the pretrained ImageNet backbones expect 3 channels. This table shows what it has to deal with.

In [ ]:
manifest.groupby(["category", "width", "height", "mode"]).size().rename("images").to_frame()

**What this shows**

- Every image is **square**, so resizing to 256 × 256 does not stretch anything.
- Bottle images are 900 px wide, carpet and screw 1024 px. Going to 256 px shrinks them about 3.5× and 4× in each direction.
- **Screw images are grayscale** (`L`). `preprocess()` copies the single channel three times, which is exactly what a 3-channel backbone needs (a unit test checks this).

## 5. What do good parts and defects look like?

One figure per category. The **first row** shows good images from `train`; **every other row** is one defect type from `test`. The red outline is the ground-truth mask: the exact pixels an expert marked as defective. Thumbnails are 256 × 256, the size the model will see.

In [ ]:
def show_examples(category: str, per_row: int = 4, thumb: int = 256) -> None:
    rows = manifest[manifest["category"] == category]
    groups = [("good (train)", rows[rows["split"] == "train"])]
    groups += list(rows[rows["label"] == 1].groupby("defect_type"))
    fig, axes = plt.subplots(len(groups), per_row, figsize=(2.3 * per_row, 2.3 * len(groups)))
    for r, (name, group) in enumerate(groups):
        for c, row in enumerate(group.sample(per_row, random_state=0).itertuples()):
            ax = axes[r, c]
            with Image.open(row.path) as image:
                ax.imshow(image.convert("RGB").resize((thumb, thumb)))
            if isinstance(row.mask_path, str):
                with Image.open(row.mask_path) as mask:
                    small = mask.resize((thumb, thumb), Image.Resampling.NEAREST)
                    ax.contour(np.asarray(small) > 0, colors="red", linewidths=0.8)
            ax.set_xticks([])
            ax.set_yticks([])
        axes[r, 0].set_ylabel(name, fontsize=9)
    fig.suptitle(category, fontsize=12)
    fig.tight_layout()
    plt.show()

### Bottle

The bottle is photographed from above, always centred in the same place. `broken_large` and `broken_small` are pieces missing from the rim; `contamination` is something inside the bottle that should not be there. The good bottles look almost identical, and the defects are large and obvious: this is the "easy" category that proves the pipeline works.

In [ ]:
show_examples("bottle")

### Carpet

A **texture**: the same woven pattern everywhere, with no fixed position. A defect can appear anywhere. `color` is a dark stain, `cut` and `hole` disturb the weave, `metal_contamination` is a tiny bright fragment and `thread` is a thin loose thread. The last two are small and thin, which makes them harder to find.

In [ ]:
show_examples("carpet")

### Screw

The same kind of screw, but **lying at a different angle and position in every image**, even among the good ones. The defects are small: a damaged tip (`manipulated_front`), scratches on the head or neck, damaged thread. Both facts make screw the hard category: the model must accept any orientation as normal, yet notice a defect that covers a tiny part of the image.

In [ ]:
show_examples("screw")

## 6. How much do good parts vary?

A one-class model has to accept every normal variation and still notice defects. To see where good parts differ from each other, the next figure takes all `train` images of a category, converts them to grayscale at 128 × 128, and shows the **standard deviation of every pixel across the images**. Dark means "the same in every good image"; bright means "this area changes from one good part to the next".

In [ ]:
def load_gray(path: str, size: int = 128) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("L").resize((size, size)), dtype=np.float32)


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, category in zip(axes, CATEGORIES, strict=True):
    is_train = (manifest["category"] == category) & (manifest["split"] == "train")
    stack = np.stack([load_gray(p) for p in manifest.loc[is_train, "path"]])
    shown = ax.imshow(stack.std(axis=0), cmap="magma", vmin=0, vmax=80)
    ax.set_title(f"{category}: pixel std over {len(stack)} good images", fontsize=9)
    ax.axis("off")
fig.colorbar(shown, ax=axes, fraction=0.02, label="std of gray level (0-255)")
plt.show()

**What this shows**

- **Bottle:** almost black. Good bottles are nearly identical; only the rings and a couple of reflections change a little. A defect stands out clearly against such a stable background.
- **Carpet:** an even, medium level everywhere. The weave shifts slightly between images, but no area is special, so the model must learn the *pattern*, not positions.
- **Screw:** a bright disc. Because the screw can point in any direction, almost every pixel in that disc is sometimes screw and sometimes background. Comparing an image pixel by pixel to an "average good screw" would fail. PatchCore copes because it compares each patch with its **nearest** normal patch from anywhere in the memory bank, wherever that patch came from.

## 7. How big are the defects?

For each defective image, the mask tells us what share of the image is defective. Small defects are the hard ones: when the image is shrunk to 256 × 256, a tiny defect becomes only a handful of pixels.

In [ ]:
def defect_percent(mask_path: str) -> float:
    with Image.open(mask_path) as mask:
        return 100 * float((np.asarray(mask) > 0).mean())


defects = manifest[manifest["label"] == 1].copy()
defects["area_pct"] = [defect_percent(p) for p in defects["mask_path"]]
summary = defects.groupby("category")["area_pct"].agg(
    images="size",
    smallest="min",
    median="median",
    largest="max",
    under_1_pct=lambda a: 100 * (a < 1).mean(),
)
summary.round(2)

`under_1_pct` is the share of defective images whose defect covers less than 1% of the image. The same numbers per defect type:

In [ ]:
defects.groupby(["category", "defect_type"])["area_pct"].agg(["min", "median", "max"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.logspace(-1.1, 1.5, 30)
for category in CATEGORIES:
    values = defects.loc[defects["category"] == category, "area_pct"]
    ax.hist(values, bins=bins, alpha=0.6, label=category)
ax.set_xscale("log")
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.text(1.05, ax.get_ylim()[1] * 0.9, "1% of the image", fontsize=8)
ax.set_xlabel("defect area, % of the image (log scale)")
ax.set_ylabel("defective images")
ax.legend()
plt.show()

**What this shows**

- **Bottle** defects are large: median 5.7% of the image, and only 5% of them are under 1%.
- **Carpet** sits in between: median 1.5%, but 42% are under 1%. The small ones are mostly `metal_contamination` (median 0.39%) and `thread` (median 0.64%).
- **Screw** defects are tiny: median 0.29%, and **99% of them are under 1%**. The whole screw distribution sits left of the dashed line.

Because small defects are so common, two choices later in the project matter: report **AUPRO** (which counts every defect region equally) next to pixel AUROC (which large defects dominate), and test **image sizes larger than 256** in Phase 2.

### What this means at the model's resolution

At 256 × 256 an image has 65,536 pixels. PatchCore compares **patches**: its `layer2` features form a 32 × 32 grid, so each grid cell summarizes an 8 × 8 = 64-pixel area. The table converts the defect sizes into pixels and grid cells at 256 px.

In [ ]:
pixels = summary[["smallest", "median"]] / 100 * 256 * 256
pd.DataFrame(
    {
        "smallest (pixels)": pixels["smallest"],
        "median (pixels)": pixels["median"],
        "smallest (8x8 cells)": pixels["smallest"] / 64,
        "median (8x8 cells)": pixels["median"] / 64,
    }
).round(1)

### Seeing it: the smallest defect in the dataset

Left: the whole image with the defect outlined. Middle: a zoom at full resolution. Right: **the same zoom in exactly what the model receives**, the output of `preprocess(image, 256)` with the normalization undone. The mask is resized with nearest-neighbour, so it stays strictly 0 or 255.

In [ ]:
def crop(array: np.ndarray, cy: int, cx: int, half: int) -> np.ndarray:
    y0, x0 = max(cy - half, 0), max(cx - half, 0)
    return array[y0 : y0 + 2 * half, x0 : x0 + 2 * half]


smallest = defects.sort_values("area_pct").iloc[0]
with Image.open(smallest["path"]) as image, Image.open(smallest["mask_path"]) as mask:
    full = np.asarray(image.convert("RGB"))
    full_mask = np.asarray(mask) > 0
    mask_256 = np.asarray(mask.resize((256, 256), Image.Resampling.NEAREST)) > 0
    model_input = preprocess(image, 256)
seen = np.clip(model_input.transpose(1, 2, 0) * STD + MEAN, 0, 1)  # undo the normalization

ys, xs = np.nonzero(full_mask)
cy, cx, half = int(ys.mean()), int(xs.mean()), 96
scale = 256 / full.shape[0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(full)
axes[0].contour(full_mask, colors="red", linewidths=0.8)
axes[0].add_patch(
    plt.Rectangle((cx - half, cy - half), 2 * half, 2 * half, fill=False, color="yellow")
)
axes[0].set_title(
    f"{smallest['category']} / {smallest['defect_type']}: {smallest['area_pct']:.3f}% of the image",
    fontsize=9,
)
axes[1].imshow(crop(full, cy, cx, half))
axes[1].contour(crop(full_mask, cy, cx, half), colors="red", linewidths=0.8)
axes[1].set_title(f"zoom at full resolution ({full.shape[0]} px)", fontsize=9)
small_c = (int(cy * scale), int(cx * scale), int(half * scale))
axes[2].imshow(crop(seen, *small_c), interpolation="nearest")
axes[2].contour(crop(mask_256, *small_c), colors="red", linewidths=0.8)
axes[2].set_title("same zoom in what the model sees (256 px)", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.show()
print(f"defect pixels: {full_mask.sum()} at full resolution, {mask_256.sum()} at 256 px")

**What this shows**

The smallest defect is a scratch on a screw's neck. At full resolution it covers about a thousand pixels and its texture is visible. At 256 px it shrinks to **65 pixels, roughly one 8 × 8 PatchCore cell**: still a brighter blob, but its detail is gone. For the median screw defect the table above gives about 3 cells. This is the strongest argument for the Phase 2 experiment with image sizes 224, 256 and 320.

## 8. Summary of findings

These go into [`docs/data_card.md`](../docs/data_card.md).

1. **1,007 images** in the manifest (647 train/good, 360 test) plus **162** in the simulation pool, with **no overlap** between the two.
2. **The test sets are small.** `holdout` has only 10 / 14 / 21 good images for bottle / carpet / screw, so one wrong decision moves the false-reject rate by 5 to 10 points. Report these numbers with that caveat.
3. **All images are square**; bottle is 900 px, carpet and screw 1024 px; **screw is grayscale**.
4. **Good parts vary differently:** bottles barely change, carpet varies evenly, screws rotate freely. Expect screw to be the hardest category.
5. **Defects are often tiny:** 99% of screw defects and 42% of carpet defects cover under 1% of the image. At 256 px the median screw defect is about 3 PatchCore cells and the smallest about 1.
6. **Consequences for the next phases:** report AUPRO next to pixel AUROC, try image sizes above 256 for screw, always resize masks with nearest-neighbour, and report results per category.